# Semantic Kernel Tool Use Example 

## Import the Needed Packages 

In [1]:
import os
import asyncio

from typing import Annotated
from openai import AsyncOpenAI


from semantic_kernel.kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.contents import ChatHistory


from semantic_kernel.agents.open_ai import OpenAIAssistantAgent
from semantic_kernel.contents import AuthorRole, ChatMessageContent
from semantic_kernel.functions import kernel_function

from semantic_kernel.connectors.ai import FunctionChoiceBehavior

from semantic_kernel.contents.function_call_content import FunctionCallContent
from semantic_kernel.contents.function_result_content import FunctionResultContent
from semantic_kernel.functions import KernelArguments, kernel_function

## Creating the Plugins    
Semantic Kernel uses plugins as tools that can be called by the agent. A plugin can have multiple `kernel_functions` in it as a group. 

In the example below, we create a `WorkoutPlugin` that has two functions: 
1. Provides a list of workout using the `get_workout` function
2. Provides a lis of availablity for each workout using the `get_availabilty` function, 

In [2]:
# Define a sample plugin for the sample
class WorkoutPlugin:
    """A List of Workout available."""

    @kernel_function(description="Provides a list of workout available.")
    def get_workout(self) -> Annotated[str, "Returns the specials from the menu."]:
        return """
        Treadmill Running/Walking,
        Cycling/Spin Classes,
        Strength Training,
        Yoga,
        Pilates,
        HIIT (High-Intensity Interval Training),
        Zumba,
        Swimming,
        Rowing,
        CrossFit.
        """

    @kernel_function(description="Provides the availability of a workout.")
    def get_availability(
        self, destination: Annotated[str, "The workout to check availability for."]
    ) -> Annotated[str, "Returns the availability of the workout."]:
        return """
        Treadmill Running/Walking - Available,
        Cycling/Spin Classes - Not Available,
        Strength Training - Not Available,
        Yoga - Available,
        Pilates - Available,
        HIIT (High-Intensity Interval Training) - Not Available,
        Zumba - Available,
        Swimming - Available,
        Rowing - Not Available,
        CrossFit - Not Available.
        """

## Creating  the client

In [3]:
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.inference.ai.azure.com/")

kernel = Kernel()
kernel.add_plugin(WorkoutPlugin(), plugin_name="workout")

service_id = "agent"

chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
    service_id=service_id
)
kernel.add_service(chat_completion_service)

## Setting the Function Choice Behavior 

In Semantic Kernel, we have the ability to have some control of the agent choice of functions. This is done by using the `FunctionChoiceBehavior` class. 

The code below sets it to `Auto` which allows the agent to choose among the available functions or not choose any. 

This can also be set to:
`FunctionChoiceBehavior.Required` - to require the agent to choose at least one function 
`FunctionChoiceBehavior.NoneInvoke` - instructs the agent to not choose any function. (good for testing)

In [4]:
settings = kernel.get_prompt_execution_settings_from_service_id(
    service_id=service_id)
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

## Creating the Agent 
Now we will create the Agent by using the Agent Name and Instructions that we can set. 

You can change these settings to see how the differences in the agent's response. 

In [5]:
AGENT_NAME = "FitnessCoach"
AGENT_INSTRUCTIONS = "Answer questions about the workouts and their availability."
# Create the agent
agent = ChatCompletionAgent(
    service_id=service_id,
    kernel=kernel,
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    arguments=KernelArguments(settings=settings),
)

## Running the Agent 

Now we wil run the AI Agent. In this snippet, we can add two messages to the `user_input` to show how the agent responds to followup questions. 

The agent should call the correct function to get the list of available workouts and confirm the availablity of a certain workout. 

You can change the `user_inputs` to see how the agent responds. 

In [6]:
async def main():
    # Define the chat history
    chat_history = ChatHistory()

    # Respond to user input
    user_inputs = [
        "What workout are available?",
        "Is Football available?",
    ]

    for user_input in user_inputs:
        # Add the user input to the chat history
        chat_history.add_user_message(user_input)
        print(f"# User: '{user_input}'")

        agent_name: str | None = None
        print("# Assistant - ", end="")
        async for content in agent.invoke_stream(chat_history):
            if not agent_name:
                agent_name = content.name
                print(f"{agent_name}: '", end="")
            if (
                not any(isinstance(item, (FunctionCallContent, FunctionResultContent))
                        for item in content.items)
                and content.content.strip()
            ):
                print(f"{content.content}", end="", flush=True)
        print("'")

await main()

# User: 'What workout are available?'
# Assistant - FitnessCoach: 'The available workouts are:

1. Treadmill Running/Walking2. Cycling/Spin Classes3. Strength Training4. Yoga5. Pilates6. HIIT (High-Intensity Interval Training)
7. Zumba8. Swimming9. Rowing10. CrossFit'
# User: 'Is Football available?'
# Assistant - FitnessCoach: 'Football is not available.Here are the availability statuses for other workouts:

- Treadmill Running/Walking - Available- Cycling/Spin Classes - Not Available- Strength Training - Not Available- Yoga - Available- Pilates - Available- HIIT (High-Intensity Interval Training) - Not Available- Zumba - Available- Swimming - Available- Rowing - Not Available- CrossFit - Not Available'
